# N132 · 组合博弈、Nim与SG函数

**目标：** 在无偏正常玩法条件下用状态异或合并游戏。

**先修：** N115, N125, N082。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 必胜/必败；mex；SG递推；Nim和；正常/反常玩法边界。

**配套讲解来源：** [同名逐章意见](../../comment/132_impartial_games_sprague_grundy.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章要解决的问题是：**两个人轮流玩一个没有平局的取子游戏，不实际下完，只看当前局面就能判断"轮到走的人是赢是输"。** 前提条件（“学习目标”部分 说的"无偏正常玩法"）有三条：双方可走的动作只取决于局面、不看人（这叫无偏）；谁没步可走谁输（这叫正常玩法，区别于"拿最后一颗的输"的反常玩法）；游戏保证会在有限步内结束。

给你具体到数字的例子（经典 Nim：一堆石子，每轮拿 1~3 根，拿到最后一根者赢）：

- 输入：`n = 4`
- 输出：`can_win_nim(4) = False`（先手必败）
- 为什么？先手无论拿 1、2、3 根，剩下 3、2、1 根，后手都能一次拿光。而 n=5 时先手拿 1 根留下 4 根给对手，对手立刻陷入刚才的必败态，所以先手必胜。规律就是"n 是 4 的倍数则输"，代码 `n%4!=0` 一行搞定。

“思路推导”部分 有个特别提醒值得先记住：**n%4 这条规律只属于"每次拿 1~3 根"这一个变体**，换成本章另一个游戏（每次拿平方数根）或改成"每次拿 1~2 根"，规律完全不同，不能乱套。

## 2. 基础知识：先读懂这些词

- **无偏博弈（impartial game）**：两个人在同一个局面下可做的选择完全相同，规则不偏袒任何一方（国际象棋不是无偏的——你只能动自己颜色的棋子）。无偏是后面 SG 异或合并成立的前提。
- **正常玩法 vs 反常玩法（normal vs misère）**：正常玩法"无步可走者输"；反常玩法（misère，失手规则）"被迫拿最后一个的输"。本章全部结论只对正常玩法成立，“正确性与复杂度”部分 明确把"失手规则"列为不能直接套模板的情形。
- **必败态 / 必胜态（P-position / N-position）**：轮到走的人必输的局面叫必败态（比如空堆）；轮到走的人存在一步制胜的局面叫必胜态。判定规则很朴素：所有后继都是必胜态 → 当前是必败态；存在一个必败态后继 → 当前是必胜态。
- **减法游戏（subtraction game）**：一堆石子、每轮从中拿走"允许集合"里的某个数量。拿 1~3 是一种，拿 1 或 2 是另一种，拿完全平方数又是一种——同一框架不同允许集合。
- **mex（最小缺失非负数，minimum excludant）**：一个非负整数集合里"第一个没出现的数"。mex{1,2,4} = 0，mex{0,1,3} = 2，mex(空集) = 0。它是 SG 值的定义材料。
- **SG 函数（Sprague–Grundy）**：给每个局面打的分：没有后继的局面 SG=0；一般局面 SG = mex(所有后继的 SG 集合)。它把"必败态"精确刻画为 SG=0 的局面，必胜态为 SG≠0。
- **Nim 和（异或合并）**：多个**互相独立**的子游戏拼成一个大游戏时，大游戏的 SG 等于各子游戏 SG 的按位异或（XOR，记 ⊕）。这就是"拆开算再合并"的依据。
- **记忆化（memoization）**：把算过的局面答案存进字典，再次遇到直接查表。`grundy` 里的 `memo` 就是干这个的，避免指数级重复展开。
- **有环检测（active 集合）**：正在递归求解中又绕回自己的局面说明游戏图有环（可能永远下不完），这不是本章前提允许的输入，立刻报错而不是瞎猜一个值。

## 3. 思路推导：从小例子开始

- **Step 1：从终点倒推。** 一切从"无步可走 = 输"出发。空石堆必败，SG(0)=0。
- **Step 2：逐个状态用 mex 打分。** 状态 x 的后继是 x 减去允许集合里每个合法取量，SG(x) = mex{后继的 SG}。
- **Step 3：读结论。** SG=0 的局面先手输，SG≠0 的局面先手赢（第五节详细解释为什么）。
- **Step 4：多个独立堆时异或合并。** 总 SG = 各堆 SG 的异或；为零则先手输，非零则先手赢。
- **Step 5：具体游戏各自找规律。** 拿 1~3：SG 周期 4，n%4==0 必败；拿 1 或 2：SG 周期 3（见下表）；拿平方数：没有简单公式，用 DP 逐位算。

**手算演示一：拿 1 或 2 的减法游戏算 SG**（这正是 “运行示例”部分 表格第二列的数据来源，允许集合 {1,2}）：

| 石子数 n | 后继 | 后继 SG 集合 | SG = mex |
|----------|------|--------------|----------|
| 0 | 无 | ∅ | 0 |
| 1 | {0} | {0} | 1 |
| 2 | {1,0} | {1,0} | 2 |
| 3 | {2,1} | {2,1} | 0 |
| 4 | {3,2} | {0,2} | 1 |
| 5 | {4,3} | {1,0} | 2 |
| 6 | {5,4} | {2,1} | 0 |

看到周期了吗：0,1,2,0,1,2……n%3==0 的局面 SG=0，先手必败。**同一个"拿石子"框架，允许集合从 {1,2,3} 换成 {1,2}，必败条件就从 n%4 变成 n%3**——这就是 “思路推导”部分 警告"不能把 n%4 误套到所有变体"的实证。

**手算演示二：“运行示例”部分 完整对照表**（三种玩法并排，n=0~10，True/False 表示先手是否必胜）：

| n | 取1或2的SG | 取1..3是否必胜 | 取平方数是否必胜 |
|---|-----------|----------------|------------------|
| 0 | 0 | False | False |
| 1 | 1 | True | True |
| 2 | 2 | True | False |
| 3 | 0 | True | True |
| 4 | 1 | False | True |
| 5 | 2 | True | False |
| 6 | 0 | True | True |
| 7 | 1 | True | False |
| 8 | 2 | False | True |
| 9 | 0 | True | True |
| 10 | 1 | True | False |

平方取石那列手算两格找感觉：n=2 时只能拿 1，剩 1 根给对手、对手拿走获胜，所以 False；n=4 时可以拿 4（2²）直接拿光获胜，所以 True。

**手算演示三：两堆独立子游戏异或合并。** 两堆各 a、b 根，每轮任选一堆拿 1 或 2 根。取 (a,b)=(4,1)：SG(4)⊕SG(1) = 1⊕1 = 0，先手必败——你可以摆一遍验证：不管先手怎么动哪一堆，后手总能在另一堆做出"对称"回应。取 (3,3)：0⊕0=0，先手也必败。取 (1,2)：1⊕2=3≠0，先手必胜（拿 2 那堆的 1 根，变成 (1,1)，SG 1⊕1=0 甩给对手）。测试里的 `win_pair(a,b)` 暴力对账覆盖了 a,b < 9 的全部 81 种组合。

## 4. 核心代码：can_win_nim

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def can_win_nim(n):
    if n < 0:
        raise ValueError('nonnegative pile required')
    return n % 4 != 0
```

### 逐段读懂代码

“分段实现”部分 有三个函数。

**函数一：`can_win_nim(n)` —— 经典 Nim（每次拿 1~3 根）。**

```python
def can_win_nim(n):
    if n < 0:
        raise ValueError('nonnegative pile required')
    return n % 4 != 0
```

- `if n<0: raise ValueError(...)`：挡住负的石子数。石子数是物理量，负数属于调用方算错，报错处理。
- `return n%4!=0`：为什么模 4？SG 序列是 0,1,2,3,0,1,2,3,…（后继恰好是连续四个里的前三个，mex 周期 4），n%4==0 即 SG=0 即必败。一句话理由：4 的倍数无论拿几根都会变成非 4 倍数，而任何非 4 倍数都能拿成 4 的倍数——主动权永远在"能把余数归零"的那一方。

**函数二：`winner_square_game(n)` —— 平方取石（每次拿完全平方数根）。**

```python
def winner_square_game(n):
    if n < 0:
        raise ValueError('nonnegative pile required')
    dp = [False] * (n + 1)
    for x in range(1, n + 1):
        dp[x] = any((not dp[x - s * s] for s in range(1, isqrt(x) + 1)))
    return dp[n]
```

- `if n<0: raise ValueError(...)`：同样的负数防御。
- `dp=[False]*(n+1)`：dp[x] 记"x 根石子时先手是否必胜"。下标 0..n，dp[0]=False（没得拿，先手输）保持初始值即可。
- `for x in range(1,n+1)`：从小到大填表。小局面先算好，大局面直接查——递推替代递归。
- `any(not dp[x-s*s] for s in range(1,isqrt(x)+1))`：一行浓缩了必胜判定。`range(1,isqrt(x)+1)` 枚举 s，s² 是不超过 x 的平方数（isqrt 保证 s²≤x，这正是 N127 用过的同一个 isqrt）；`x-s*s` 是拿走 s² 根后的剩余局面；`not dp[...]` 即"后继是必败态"。`any(...)` 的含义："存在一个后继是必败态"——存在则当前必胜，全不存在则 any 返回 False，当前必败。
- `return dp[n]`：查表收工。

**函数三：`grundy(state, moves)` —— 通用 SG 值计算（带记忆化和环检测）。**

```python
def grundy(state, moves):
    memo = {}
    active = set()

    def solve(s):
        if s in memo:
            return memo[s]
        if s in active:
            raise ValueError('finite acyclic game graph required')
        active.add(s)
        reachable = {solve(t) for t in moves(s)}
        active.remove(s)
        value = 0
        while value in reachable:
            value += 1
        memo[s] = value
        return value
    return solve(state)
```

- `memo={}; active=set()`：memo 是记忆化字典（算过的状态直接查）；active 是"正在递归栈里的状态"集合，专门用来抓环。
- `if s in memo: return memo[s]`：查表命中，直接返回，省掉重复展开。
- `if s in active: raise ValueError(...)`：正在算 s 的途中又遇到了 s，说明沿这条链转回来了——游戏图有环、可能永远下不完，违背本章"有限终止"前提。报错拒绝而不是返回 0（返回 0 会把环错误地当成必败态，测试专门验证了这点）。
- `active.add(s); reachable={solve(t) for t in moves(s)}; active.remove(s)`：进栈标记、递归求所有后继的 SG（`moves(s)` 是调用方传入的"后继生成函数"，比如拿 1 或 2 就是 `[n-k for k in (1,2) if k<=n]`）、出栈撤销标记。集合推导式一口气递归+去重。
- `value=0; while value in reachable: value+=1`：手写 mex——从 0 开始往上找第一个不在后继 SG 集合里的数。
- `memo[s]=value; return value`：存表、返回。`return solve(state)` 从指定初始状态启动。

## 5. 分段实现：按顺序运行

**本章接口：** `can_win_nim(n)`、`winner_square_game(n)`、`grundy(state, moves)`

### 导入本章使用的库

In [1]:
from math import isqrt

### can_win_nim

In [2]:
def can_win_nim(n):
    if n < 0:
        raise ValueError('nonnegative pile required')
    return n % 4 != 0

### winner_square_game

In [3]:
def winner_square_game(n):
    if n < 0:
        raise ValueError('nonnegative pile required')
    dp = [False] * (n + 1)
    for x in range(1, n + 1):
        dp[x] = any((not dp[x - s * s] for s in range(1, isqrt(x) + 1)))
    return dp[n]

### grundy

In [4]:
def grundy(state, moves):
    memo = {}
    active = set()

    def solve(s):
        if s in memo:
            return memo[s]
        if s in active:
            raise ValueError('finite acyclic game graph required')
        active.add(s)
        reachable = {solve(t) for t in moves(s)}
        active.remove(s)
        value = 0
        while value in reachable:
            value += 1
        memo[s] = value
        return value
    return solve(state)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

moves=lambda n:[n-k for k in (1,2) if k<=n]
show_table(['石子数','取1或2的SG','取1..3是否必胜','取平方数是否必胜'],[(n,grundy(n,moves),can_win_nim(n),winner_square_game(n)) for n in range(11)])

石子数,取1或2的SG,取1..3是否必胜,取平方数是否必胜
0,0,False,False
1,1,True,True
2,2,True,False
3,0,True,True
4,1,False,True
5,2,True,False
6,0,True,True
7,1,True,False
8,2,False,True
9,0,True,True


## 7. 正确性、适用条件与复杂度

SG为0时，所有后继SG都非零；SG非零时，mex定义保证存在SG为0的后继，因此输赢与SG是否为0一致。独立游戏和的异或为0时任一单游戏变化使异或非零；异或非零时可选最高不同位对应的分量转移到较小的指定SG，使总异或变0。该结论依赖正常玩法、无偏与有限终止。

**代价：** 取1..3 O(1)算术；平方取石O(n√n)时间O(n)空间；一般SG按可达DAG求值，O(V+E)级集合操作和O(V+E)空间上界。递归版仅用于小有限状态；有环游戏、失手规则和双方动作不同的游戏不能直接套此模板。

### 展开理解

**SG=0 为什么恰好等于必败？** 两个方向各说一句。若某局面 SG=0，按 mex 的定义，0 不在后继 SG 集合里，也就是**每一个**后继的 SG 都不是 0——你每走一步都把对手送进必胜态（非零态）。若某局面 SG=g≠0，mex 的定义保证 0..g−1 里每个数都在后继集合中，特别地**存在**一个 SG=0 的后继——你走这一步就把对手按进了必败态。一句话：零态只能走向非零态，非零态总能走向零态，谁被按在零态谁输。

**多堆异或为什么成立？** 关键是两条。第一条：如果各堆 SG 的异或是 0，那么你从任何一堆拿走一些（只动一堆，SG 值从 g 变成 g′≠g），新的总异或必然不再是 0——因为只有一个分量变了，异或的公共抵消被破坏。于是你**不得不**把非零局面送给对手。第二条：如果异或是 g≠0，取 g 的最高位 1，必有某堆的 SG 在该位也是 1（否则这一位异或不出来 1）；设这堆 SG 是 gₖ，令目标值 t = g ⊕ gₖ，可以验证 t < gₖ（最高位被消掉了，剩下的位只会更小），于是这堆里存在走法把 SG 降到 t，走完后总异或变成 0。先手每次都做"归零"操作，游戏有限步内结束，最后一步必然是"对手面对全零、无步可走"。这两条合起来就是：异或为零先手输，非零先手赢。

**复杂度（结合具体数字）：** `can_win_nim` 是 O(1) 的一次取模，n = 10⁹ 也瞬间。`winner_square_game` 是 O(n√n)：每个 x 要试 √x 个平方数，n = 10⁵ 时约 10⁵×316 ≈ 3×10⁷ 次查表，秒级完成；空间 O(n)。通用 `grundy` 相当于在游戏图上做遍历，把每个状态、每条转移边碰一遍，O(V+E) 的集合操作和空间；“正确性与复杂度”部分 提醒递归版只适合小规模有限状态，图一大应改成显式 DP 或迭代。

**边界警示（“正确性与复杂度”部分 原话的翻译）：** 这套异或模板依赖三个前提——正常玩法（拿最后一颗赢）、无偏（双方动作相同）、有限终止（没有环）。反常玩法（最后一颗输）、有环游戏、双方动作不同的游戏（比如棋类）都不能直接套，套了答案就是错的。

## 8. 单元测试：每个用例在检查什么

1. `assert not can_win_nim(4) and can_win_nim(5)` —— **正常值 + 分界点**：n=4 是第一个必败点（4 的倍数规律的首个见证），n=5 验证"必败点 +1 立即翻盘"。
2. `assert not winner_square_game(2)` —— **小必败例**：2 根只能拿 1，送对手 1 根被反杀，这是平方取石最小的必败局面。
3. `assert grundy(0,lambda s:[])==0` —— **终点状态**：后继函数恒返回空列表（没有后继），mex(空)=0，验证 SG 定义的起点。
4. 双堆异或 vs 暴力对账：`win_pair(a,b)` 是直接按必胜定义写的朴素递归（任选一堆拿 1 或 2，存在后继必败即胜，`@cache` 只为加速不改变语义），对 a,b < 9 的 81 个组合断言 `bool(grundy(a)^grundy(b))==win_pair(a,b)`——SG 异并理论和小规模真值表逐格吻合，这是本节最有分量的测试。
5. 平方取石 vs 暴力对账：`square_ref(n)` 同样是按定义写的朴素递归（拿任何平方数 k²，存在必败后继即胜），对 n < 80 逐一对照 `winner_square_game`——DP 版和定义版必须步调一致。
6. 环检测测试：`moves=lambda s:[s]` 的后继永远是自己，`try/except` 要求 `grundy` 必须抛 ValueError；若它悄悄返回了值，就 `raise AssertionError('cycle should not be silently treated as a losing state')`——保证有环输入被明说拒绝，而不是被当成必败态蒙混过关。

In [6]:
assert not can_win_nim(4) and can_win_nim(5)

assert not winner_square_game(2)

assert grundy(0, lambda s: []) == 0

from functools import cache

moves = lambda n: [n - k for k in (1, 2) if k <= n]

@cache
def win_pair(a, b):
    return any((not win_pair(a - k, b) for k in (1, 2) if a >= k)) or any((not win_pair(a, b - k) for k in (1, 2) if b >= k))

for a in range(9):
    for b in range(9):
        assert bool(grundy(a, moves) ^ grundy(b, moves)) == win_pair(a, b)

@cache
def square_ref(n):
    return any((not square_ref(n - k * k) for k in range(1, isqrt(n) + 1)))

for n in range(80):
    assert winner_square_game(n) == square_ref(n)

try:
    grundy(0, lambda s: [s])
except ValueError:
    pass
else:
    raise AssertionError('cycle should not be silently treated as a losing state')

print('N132: 所有本章断言通过')

N132: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 用小状态完整搜索验证Nim规律。

**练习 2：** 给出不满足无偏条件不能直接异或的游戏。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（小状态完整搜索验证 Nim 规律）：** 提示——对"每次拿 1~3 根"的单堆游戏写一个暴力递归 `win(n) = any(not win(n-k) for k in (1,2,3) if k<=n)`，从 n=0 打表打到 n=16 左右，观察 False 出现的位置是不是恰好 0,4,8,12,16。再把两堆的情形 `win2(a,b)`（每轮只从一堆里拿 1~3 根）写成朴素递归打一张小表，对照理论值：每堆的 SG 就是 n%4（可用 `grundy` 配 `lambda n:[n-k for k in (1,2,3) if k<=n]` 现算），两堆合并的胜负等于两堆 SG 的异或是否非零——暴力表和异或预言必须逐格一致。写清楚边界：n=0 和"两堆都空"时轮到谁走谁就输。

**练习 2（构造不满足无偏条件的反例）：** 提示——找一个"双方可走动作不同"的玩具游戏，比如：一堆 3 根石子，先手只能拿奇数根（1 或 3），后手只能拿偶数根（2），拿光者赢。这时"后继集合依赖轮到谁"，SG 函数吃不到"轮到谁"这个信息，直接异并必然算错——你可以定义状态为 (剩根数, 轮到谁) 写朴素递归求真值，再故意用单状态 SG 异或算一遍，展示两者在小例子上的分歧。手算 3 根的完整博弈树作示例，说清输入输出和边界（0 根且轮到拿的人即输）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
from math import isqrt

def can_win_nim(n):
    if n < 0:
        raise ValueError('nonnegative pile required')
    return n % 4 != 0

def winner_square_game(n):
    if n < 0:
        raise ValueError('nonnegative pile required')
    dp = [False] * (n + 1)
    for x in range(1, n + 1):
        dp[x] = any((not dp[x - s * s] for s in range(1, isqrt(x) + 1)))
    return dp[n]

def grundy(state, moves):
    memo = {}
    active = set()

    def solve(s):
        if s in memo:
            return memo[s]
        if s in active:
            raise ValueError('finite acyclic game graph required')
        active.add(s)
        reachable = {solve(t) for t in moves(s)}
        active.remove(s)
        value = 0
        while value in reachable:
            value += 1
        memo[s] = value
        return value
    return solve(state)

# 示例与回归测试
assert not can_win_nim(4) and can_win_nim(5)

assert not winner_square_game(2)

assert grundy(0, lambda s: []) == 0

from functools import cache

moves = lambda n: [n - k for k in (1, 2) if k <= n]

@cache
def win_pair(a, b):
    return any((not win_pair(a - k, b) for k in (1, 2) if a >= k)) or any((not win_pair(a, b - k) for k in (1, 2) if b >= k))

for a in range(9):
    for b in range(9):
        assert bool(grundy(a, moves) ^ grundy(b, moves)) == win_pair(a, b)

@cache
def square_ref(n):
    return any((not square_ref(n - k * k) for k in range(1, isqrt(n) + 1)))

for n in range(80):
    assert winner_square_game(n) == square_ref(n)

try:
    grundy(0, lambda s: [s])
except ValueError:
    pass
else:
    raise AssertionError('cycle should not be silently treated as a losing state')

print('N132: 所有本章断言通过')

N132: 所有本章断言通过


## 11. 代表题与迁移

- **292. Nim Game（Nim 游戏）**：一堆石子每次拿 1~3——`can_win_nim` 的原型题（canonical），练的是从博弈倒推出 n%4 规律而非枚举。
- **1510. Stone Game IV（石子游戏 IV）**：每次拿完全平方数——`winner_square_game` 的原型题（canonical），练"必胜判定转 DP：存在必败后继则胜"。
- **810. Chalkboard XOR Game（黑板异或游戏）**：异或背景的双方博弈——作为 extension，练的是把本章的异或工具放到具体规则下重新推导，而不是直接套 SG 模板（它恰好有"擦数"这种双方动作相同但胜负条件特殊的结构）。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。